# Stage 2: Multi-agent Supervisor (LangGraph built in supervisor)


In [1]:
# %pip -q install langchain langchain-openai langgraph langgraph-supervisor python-dotenv

## 1) Imports

In [2]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph_supervisor import create_supervisor
from typing import Any, Dict, Optional
from langchain_core.tools import tool
import os
from dotenv import load_dotenv

## 2) Load environment variables - please read instructions carefully

In [3]:
# if you are running in local, uncomment below line. also make sure you shall have a .env file
load_dotenv()

True

In [4]:
# if you are running in google colab, uncomment below line. and replace "Your_API_Key" with your own openAI API key
#os.environ["OPENAI_API_KEY"] = "Your_API_Key"

## 3) Helper: pretty_print()

In [5]:
def pretty_print(response):
    for m in response["messages"]:
        print("\n---", m.type, "---")
        print(m)

    last_msg = response["messages"][-1]

    if isinstance(last_msg.content, list):
        text = "".join(
            block["text"]
            for block in last_msg.content
            if block.get("type") == "text"
        )
    else:
        text = last_msg.content
    print(text)

## 4) Local tool: estimate_trip_cost

In [6]:
@tool
def estimate_trip_cost(
    destination: str,
    days: int,
    travelers: int,
    comfort: str = "mid",
) -> Dict[str, Any]:
    """
    Estimate a rough trip budget (SGD) using simple heuristics.
    comfort: budget | mid | premium
    Returns a breakdown and total estimate in SGD.
    """
    if days <= 0 or travelers <= 0:
        raise ValueError("days and travelers must be > 0")

    comfort = comfort.lower().strip()
    if comfort not in {"budget", "mid", "premium"}:
        raise ValueError("comfort must be one of: budget, mid, premium")

    # Very rough per-person-per-day estimates (SGD) excluding flights
    lodging_per_person_per_day = {"budget": 60, "mid": 140, "premium": 300}[comfort]
    food_per_person_per_day = {"budget": 30, "mid": 60, "premium": 120}[comfort]
    local_transport_per_person_per_day = {"budget": 10, "mid": 20, "premium": 50}[comfort]
    activities_per_person_per_day = {"budget": 20, "mid": 50, "premium": 120}[comfort]

    lodging = lodging_per_person_per_day * travelers * days
    food = food_per_person_per_day * travelers * days
    transport = local_transport_per_person_per_day * travelers * days
    activities = activities_per_person_per_day * travelers * days

    subtotal = lodging + food + transport + activities
    contingency = round(subtotal * 0.12)  # 12% buffer
    total = subtotal + contingency

    return {
        "destination": destination,
        "days": days,
        "travelers": travelers,
        "comfort": comfort,
        "currency": "SGD",
        "breakdown": {
            "lodging": lodging,
            "food": food,
            "local_transport": transport,
            "activities": activities,
            "contingency": contingency,
        },
        "total_estimate": total,
        "note": "Heuristic estimate excludes international flights/insurance/visa fees.",
    }



## 5) Create specialist agents

In [7]:
RESEARCH_SYSTEM = """You are ResearchAgent.
Your ONLY job: gather factual, practical travel info using web search.
Rules:
- Use web search when needed.
- Do NOT estimate or compute cost.
- Output ONLY 5 bullets max.
- Each bullet: destination + why + travel time + one practical note.
"""

research_agent = create_agent(
    model=ChatOpenAI(model="gpt-4.1-mini", temperature=0.2),
    tools=[{"type": "web_search_preview"}],
    system_prompt=RESEARCH_SYSTEM,
    name="research_agent"
)

In [8]:
COST_SYSTEM = """You are CostAgent.
Your ONLY job: compute total cost.
Rules:
- Never invent numbers, use tools avaiable.
- If destination/days/travelers/comfort are missing, ask for them (max 2 questions) and stop.
- If user says 'add an additional one-day trip', treat it as +1 day ONLY IF baseline days is known.
- Return: total cost + short assumptions.
"""

cost_agent = create_agent(
    model=ChatOpenAI(model="gpt-4.1-mini", temperature=0.1),
    tools=[estimate_trip_cost],
    system_prompt=COST_SYSTEM,
    name="cost_agent",
)

In [9]:
ITINERARY_SYSTEM = """You are ItineraryAgent.
Your job: produce the final itinerary and incorporate research notes + cost breakdown.
Rules:
- Do NOT invent numeric costs; only use cost_breakdown if present.
- If cost_breakdown missing, ask for required info (max 2 questions).
Output format:
1) Day-by-day plan (brief)
2) Total cost (SGD) + assumptions
"""

itinerary_agent = create_agent(
    model=ChatOpenAI(model="gpt-4.1-mini", temperature=0.4),
    tools=[],
    system_prompt=ITINERARY_SYSTEM,
    name="itinerary_agent",
)

## 6) Create supervisor workflow

In [10]:
SUPERVISOR_SYSTEM = """You are a supervisor agent for travel planning. You coordinate a group of agents to plan travel.

Your job:
- collect travel requirement
- research based on requirement
- cost estimation
- generate itinerary

you have access to:
- research_agent: gather factual, practical travel info using web search
- cost_agent: estimate total travel cost
- itinerary_agent: generate day by day itinerary

Rules:
- Ask clarifying questions only when essential info is missing.
- Do not invent facts.

Output format:
1) Day-by-day plan (brief)
2) Total cost (with assumptions)
"""

workflow = create_supervisor(
    [research_agent, cost_agent,itinerary_agent],
    model=ChatOpenAI(model="gpt-4o"),
    prompt=SUPERVISOR_SYSTEM
)

## 7) Demo

In [12]:
# Compile and run
app = workflow.compile()
user_prompt = "Plan a 2-day Tokyo trip for 2 adults, mid comfort. We like food and anime. Avoid packed schedules."

result = app.invoke({"messages": [{"role": "user", "content": user_prompt}]})
pretty_print(result)


--- human ---
content='Plan a 2-day Tokyo trip for 2 adults, mid comfort. We like food and anime. Avoid packed schedules.' additional_kwargs={} response_metadata={} id='f5060345-c795-4244-ba51-935c969f9ea7'

--- ai ---
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 227, 'total_tokens': 240, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f38e8ce955', 'id': 'chatcmpl-DypJPjWCMVQpF9yKGIBV8w4hIt9xu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} name='supervisor' id='lc_run--019f3a49-1224-77a2-aefc-a9012994ca31-0' tool_calls=[{'name': 'transfer_to_itinerary_agent', 'args': {}, 'id': 'call_2xxEybWZb1ymYC1PYA0HwD1n', 'type': 'tool_call'}] inv